# Bayes' Theorem: The Disease-Test Example

This notebook introduces **Bayes' theorem** through its most famous application: interpreting a positive medical test for a rare disease. The result is one of the most counter-intuitive in all of probability — a positive result from a 99%-effective test can still mean you are *probably healthy*. It is based on the tenth video of Week 1 (Introduction to Probability).


## 1. Introduction

Imagine there is a rare disease going around and you get tested. The doctor's test is "very effective — correct most of the time." The doctor calls: **you tested positive.** Should you panic?

Before panicking, do some math: compute the probability that you actually have the disease **given** that you tested positive. This is Bayes' theorem in action.

Bayes' theorem is one of the most important theorems in probability and is used all over machine learning — spam recognition, speech detection, medical diagnosis, and much more.

After completing this notebook you will be able to:

- Break a population into the four groups defined by (sick/healthy) × (diagnosed sick/healthy).
- Compute $P(\text{sick} \mid \text{diagnosed sick})$ by counting, by probability tree, and by the Bayes formula.
- Explain *why* rare conditions make false positives dominate true positives.


## 2. Theory

### The setup

- Population: **1,000,000** people.
- The illness affects **1 in every 10,000** people → 100 sick, 999,900 healthy.
- The test is **99% effective**, meaning two things:
  - of every 100 **sick** people, 99 test sick and 1 is missed (tests healthy);
  - of every 100 **healthy** people, 1 is misdiagnosed as sick and 99 test healthy.

### Bayes' theorem

We want the probability of being sick given a positive diagnosis. Using conditional probability (notebook 08),

$$
P(S \mid +) \;=\; \frac{P(S \cap +)}{P(+)} \;=\; \frac{P(S)\,P(+ \mid S)}{P(S)\,P(+ \mid S) + P(S')\,P(+ \mid S')}
$$

The numerator is the *true positives*; the denominator is *everyone who tests positive* — true positives plus false positives. Bayes' theorem is nothing more than the general product rule applied to both branches of the probability tree from notebook 09.

### Naming the pieces

Every factor in the formula has a standard name — one general Bayesian name, and one medical name used for diagnostic tests. Written with labels:

$$
\underbrace{P(S \mid +)}_{\text{posterior}} \;=\; \frac{\overbrace{P(S)}^{\text{prior}} \cdot \overbrace{P(+ \mid S)}^{\text{likelihood}}}{\underbrace{P(+)}_{\text{evidence}}}
$$

| Symbol | Bayesian name | Medical name | Meaning | Our value |
|---|---|---|---|---|
| $P(S)$ | **prior** | prevalence | belief *before* the test: fraction of the population that is sick | $1/10{,}000 = 0.0001$ |
| $P(+ \mid S)$ | **likelihood** | sensitivity | probability a *sick* person tests positive | $0.99$ |
| $P(+ \mid S')$ | — | false-positive rate ($= 1 -$ specificity) | probability a *healthy* person tests positive | $0.01$ |
| $P(+)$ | **evidence** | — | overall probability of a positive test, sick or not; the normalizer | $10{,}098/1{,}000{,}000$ |
| $P(S \mid +)$ | **posterior** | positive predictive value | updated belief *after* seeing a positive test | $\approx 0.0098$ |

Read this way, Bayes' theorem is an **update rule**:

$$
\text{posterior} \;=\; \text{prior} \times \frac{\text{likelihood}}{\text{evidence}}
$$

The fraction $P(+ \mid S)/P(+) \approx 0.99/0.0101 \approx 98$ is the *update factor*: a positive test makes "sick" about 98 times more likely than it was before. But 98 times a tiny prior of $0.0001$ is still only $\approx 0.0098$ — **a strong test cannot rescue a very small prior.**

### Why intuition fails

"The test is 99% accurate" describes $P(+ \mid S)$ — the probability of a positive test *given* sickness. But the question you care about is the reverse: $P(S \mid +)$. When the disease is rare, the tiny 1% error rate applied to the *huge* healthy group produces far more false positives than there are sick people in total.


## 3. Worked Example: counting the four groups

Test all 1,000,000 people and sort them:

| | Diagnosed sick | Diagnosed healthy | Total |
|---|---|---|---|
| **Sick** | 99 (true positive) | 1 (missed) | 100 |
| **Healthy** | 9,999 (false positive) | 989,901 (correct) | 999,900 |
| **Total** | **10,098** | 989,902 | 1,000,000 |

You tested positive, so you are one of the **10,098** people in the left column. Of those, only **99** are actually sick:

$$
P(S \mid +) = \frac{99}{99 + 9{,}999} = \frac{99}{10{,}098} \approx 0.0098
$$

**Less than 1%.** Astonishing: you were diagnosed sick, yet the probability that you are actually sick is under 1% — because the 1% of 999,900 healthy people (9,999 false positives) vastly outnumbers the 99 true positives.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)


### Counting the four groups


In [ ]:
population = 1_000_000
prevalence = 1 / 10_000     # P(sick)
effectiveness = 0.99        # P(correct diagnosis), for sick and healthy alike

sick = population * prevalence
healthy = population - sick

true_positives = sick * effectiveness
false_negatives = sick * (1 - effectiveness)
false_positives = healthy * (1 - effectiveness)
true_negatives = healthy * effectiveness

print(f"Sick & diagnosed sick (true positive):     {true_positives:>10,.0f}")
print(f"Sick & diagnosed healthy (missed):         {false_negatives:>10,.0f}")
print(f"Healthy & diagnosed sick (false positive): {false_positives:>10,.0f}")
print(f"Healthy & diagnosed healthy:               {true_negatives:>10,.0f}")

p_sick_given_positive = true_positives / (true_positives + false_positives)
print(f"\nP(sick | diagnosed sick) = {true_positives:.0f} / "
      f"{true_positives + false_positives:.0f} = {p_sick_given_positive:.4f}")


### The same result from the Bayes formula

The function below is the update rule from section 2, with the parameters named after their roles: `p_sick` is the **prior** (prevalence), `p_pos_given_sick` the **likelihood** (sensitivity), `p_pos_given_healthy` the false-positive rate, the computed `denominator` the **evidence** $P(+)$, and the return value the **posterior**.


In [ ]:
def bayes_posterior(p_sick, p_pos_given_sick, p_pos_given_healthy):
    numerator = p_sick * p_pos_given_sick
    denominator = numerator + (1 - p_sick) * p_pos_given_healthy
    return numerator / denominator

posterior = bayes_posterior(p_sick=1 / 10_000,
                            p_pos_given_sick=0.99,
                            p_pos_given_healthy=0.01)
print(f"P(S | +) = {posterior:.4f}  (less than 1%)")


### Checking with a simulation

Simulate a million random people, test them, and look at the sick fraction *among those who tested positive* — conditioning by filtering, as in notebook 08.


In [ ]:
n = 1_000_000
is_sick = rng.random(n) < 1 / 10_000
correct = rng.random(n) < 0.99
tests_positive = np.where(is_sick, correct, ~correct)

simulated = is_sick[tests_positive].mean()
print(f"People testing positive: {tests_positive.sum():,}")
print(f"Simulated P(sick | positive): {simulated:.4f}  (theory {posterior:.4f})")


## 5. Visualizations

### Who is in the "tested positive" group?

The bar shows everyone who was diagnosed sick. Almost all of them are healthy people caught by the 1% error rate.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))

ax.barh(["Diagnosed\nsick"], [9999], color="#f28e2b", label="Healthy (false positives): 9,999")
ax.barh(["Diagnosed\nsick"], [99], left=[9999], color="#e15759", label="Sick (true positives): 99")

ax.set_xlabel("People")
ax.set_title("Of 10,098 positive tests, only 99 are actually sick (< 1%)")
ax.legend(loc="center right")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()


### The probability tree

The same four groups as a tree, as shown in the video: multiply along each path (general product rule), then compare the two "diagnosed sick" leaves.


In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 6))

root = (0, 3)
level1 = {"sick": (3.5, 4.6), "healthy": (3.5, 1.4)}
level2 = {
    "sick, diagnosed sick": (7.5, 5.3),
    "sick, diagnosed healthy": (7.5, 3.9),
    "healthy, diagnosed sick": (7.5, 2.1),
    "healthy, diagnosed healthy": (7.5, 0.7),
}
counts = {
    "sick, diagnosed sick": "99",
    "sick, diagnosed healthy": "1",
    "healthy, diagnosed sick": "9,999",
    "healthy, diagnosed healthy": "989,901",
}

def branch(a, b, label):
    ax.plot([a[0], b[0]], [a[1], b[1]], color="gray", linewidth=1.5)
    mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
    ax.annotate(label, (mx, my), textcoords="offset points", xytext=(0, 7),
                ha="center", fontsize=10, color="darkblue")

branch(root, level1["sick"], "1/10,000")
branch(root, level1["healthy"], "9,999/10,000")
branch(level1["sick"], level2["sick, diagnosed sick"], "0.99")
branch(level1["sick"], level2["sick, diagnosed healthy"], "0.01")
branch(level1["healthy"], level2["healthy, diagnosed sick"], "0.01")
branch(level1["healthy"], level2["healthy, diagnosed healthy"], "0.99")

ax.plot(*root, "o", markersize=10, color="black")
ax.annotate("1,000,000\npeople", root, textcoords="offset points",
            xytext=(-12, 0), ha="right", fontsize=10)

for name, pos in level1.items():
    ax.plot(*pos, "o", markersize=9, color="#4e79a7")
    ax.annotate(name, pos, textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=10)

for name, pos in level2.items():
    positive = "diagnosed sick" in name
    ax.plot(*pos, "s", markersize=9, color="#e15759" if positive else "#bbbbbb")
    ax.annotate(f"{name}: {counts[name]}", pos, textcoords="offset points",
                xytext=(12, 0), ha="left", fontsize=10,
                fontweight="bold" if positive else "normal")

ax.set_xlim(-2.2, 11.5)
ax.set_ylim(0, 6.2)
ax.axis("off")
ax.set_title("You tested positive → you are in one of the two red groups: 99 vs 9,999")
plt.show()


### How the answer depends on how rare the disease is

The posterior $P(S \mid +)$ climbs as prevalence rises: the rarer the disease, the less a positive test means. With a 99%-effective test, the posterior only passes 50% when about 1 in 100 people are sick.


In [ ]:
prevalences = np.logspace(-5, -0.5, 200)
posteriors = [bayes_posterior(p, 0.99, 0.01) for p in prevalences]

plt.figure(figsize=(9, 5))
plt.semilogx(prevalences, posteriors)
plt.axvline(1 / 10_000, color="red", linestyle="--",
            label="our example: 1 in 10,000 → posterior < 1%")
plt.axhline(0.5, color="gray", linestyle=":")
plt.title("P(sick | positive) vs. disease prevalence, 99%-effective test")
plt.xlabel("P(sick) — prevalence (log scale)")
plt.ylabel("P(sick | positive)")
plt.grid(True, alpha=0.3, which="both")
plt.legend()
plt.show()


## 6. Exercises

### Basic

1. In the example, what is $P(\text{healthy} \mid \text{diagnosed sick})$?
2. What is $P(\text{sick} \mid \text{diagnosed healthy})$ — the probability that a person who tested healthy is actually sick?

### Intermediate

3. Recompute $P(S \mid +)$ if the disease affects 1 in 100 people (same 99% test). How does the answer change?
4. Recompute $P(S \mid +)$ for the original 1-in-10,000 disease but a **99.99%**-effective test.

### Advanced

5. You tested positive, and the doctor orders a **second, independent test** — which also comes back positive. Use your posterior from the first test (≈ 0.0098) as the new prior and apply Bayes again. What is the probability of being sick after two positives? Verify by simulation.


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  P(healthy | +) = 9999/10098 =", round(9999 / 10098, 4))

# Exercise 2: missed sick people among all who tested healthy
p = 1 / (1 + 989_901)
print(f"2.  P(sick | -) = 1/989,902 = {p:.8f} — a negative test is very reliable.")

# Exercise 3: more common disease
print("3.  P(S | +) at 1-in-100 prevalence:",
      round(bayes_posterior(0.01, 0.99, 0.01), 4))

# Exercise 4: better test
print("4.  P(S | +) with a 99.99% test:",
      round(bayes_posterior(1 / 10_000, 0.9999, 0.0001), 4))

# Exercise 5: two positive tests — update twice
after_one = bayes_posterior(1 / 10_000, 0.99, 0.01)
after_two = bayes_posterior(after_one, 0.99, 0.01)

n = 20_000_000
is_sick = rng.random(n) < 1 / 10_000
pos1 = np.where(is_sick, rng.random(n) < 0.99, rng.random(n) < 0.01)
pos2 = np.where(is_sick, rng.random(n) < 0.99, rng.random(n) < 0.01)
both = pos1 & pos2
simulated = is_sick[both].mean()
print(f"5.  After two positives: {after_two:.4f} "
      f"(simulated {simulated:.4f} from {both.sum():,} double-positives)")


## 8. Conclusion

- Bayes' theorem reverses a conditional: from the known $P(+ \mid S)$ (test behavior) to the wanted $P(S \mid +)$ (your actual risk):

$$
P(S \mid +) = \frac{P(S)\,P(+ \mid S)}{P(S)\,P(+ \mid S) + P(S')\,P(+ \mid S')}
$$

- In Bayesian vocabulary: the **posterior** equals the **prior** (here: the prevalence) times the **likelihood** (here: the test's sensitivity), divided by the **evidence** (the overall probability of a positive test).
- For a rare disease (1 in 10,000) and a 99% test, $P(S \mid +) \approx 0.98\%$ — the false positives from the enormous healthy group swamp the true positives. A strong likelihood cannot rescue a very small prior.
- The four-group table, the probability tree, and the formula are three views of the same computation.
- Each new piece of evidence updates the probability again: after the first positive test, the posterior ($\approx 0.0098$) becomes the **prior** for the second test (exercise 5). This *prior → posterior* loop is the heart of Bayesian reasoning in machine learning.
